# 支持向量机（SVM）：寻找最优分界线

本 notebook 演示支持向量机（Support Vector Machine, SVM）的基本使用方式。

内容分为两部分：

1. 用二维示例直观理解 SVM 如何寻找“最优分界线”；
2. 使用 `predictive_maintenance.csv` 建立一个工业设备故障预测模型。

## 数据集说明

目标变量：

- `Machine failure`：`1` 表示设备故障，`0` 表示正常。

输入特征：

- `Type`：设备类型；
- `Air temperature`：空气温度；
- `Process temperature`：过程温度；
- `Rotational speed`：转速；
- `Torque`：扭矩；
- `Tool wear`：刀具磨损。

`TWF`、`HDF`、`PWF`、`OSF`、`RNF` 是故障原因/故障模式指示变量，直接作为输入特征会造成标签泄露，因此不使用。

## 1. SVM 的核心思想

对于二分类问题，SVM 希望找到一个能把两类数据分开的边界。

在二维空间中，这个边界是一条直线；在三维空间中是一个平面；在更高维特征空间中，通常称为**超平面**。

如果能把数据完全分开的线有很多条，SVM 会选择其中“最稳”的一条：

- 不仅要求分开两类样本；
- 还要求边界到两侧最近样本的距离尽可能大；
- 这个距离叫作**间隔（margin）**。

离边界最近的训练样本叫作**支持向量（support vectors）**。它们决定了最终边界的位置；其他离边界较远的样本对边界影响较小。

### 参数 C 的作用

真实数据通常无法被完美分开，所以 SVM 会允许少量样本越界或误分类。

- `C` 较小：更重视大间隔，允许更多误分类，模型更简单；
- `C` 较大：更重视减少训练误分类，间隔可能更窄，模型更复杂，容易过拟合。

因此，SVM 的“最优分界线”可以理解为：**在分类错误代价和间隔大小之间取得平衡的分界线**。

## 2. 导入库

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_blobs
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

## 3. 二维示例：直观看到 SVM 分界线

先生成两组二维数据，再训练一个线性 SVM。

下面图中：

- 黑色实线：SVM 的决策边界；
- 灰色虚线：间隔边界；
- 黄色圆圈：支持向量。

In [ ]:
X_demo, y_demo = make_blobs(
    n_samples=100,
    centers=[(2.0, 2.0), (-2.0, 6.0)],
    cluster_std=0.9,
    random_state=33,
)

demo_svm = SVC(kernel="linear", C=1.0)
demo_svm.fit(X_demo, y_demo)

x_min, x_max = X_demo[:, 0].min() - 1, X_demo[:, 0].max() + 1
y_min, y_max = X_demo[:, 1].min() - 1, X_demo[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400),
)

Z = demo_svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

In [ ]:
plt.figure(figsize=(7, 6))

plt.contour(
    xx,
    yy,
    Z,
    levels=[-1, 0, 1],
    colors=["gray", "black", "gray"],
    linestyles=["--", "-", "--"],
)

plt.scatter(
    X_demo[:, 0],
    X_demo[:, 1],
    c=y_demo,
    cmap="bwr",
    edgecolors="black",
    s=55,
)

plt.scatter(
    demo_svm.support_vectors_[:, 0],
    demo_svm.support_vectors_[:, 1],
    s=160,
    facecolors="none",
    edgecolors="gold",
    linewidths=2,
    label="Support vectors",
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Linear SVM: decision boundary and margin")
plt.legend()
plt.show()

黑色实线不是随便画的一条分界线，而是让两侧间隔尽可能大的分界线。

如果移动这条线，虽然仍可能把当前训练样本分开，但距离边界最近的样本会更近，模型对新数据的容错空间会变小。SVM 因此更倾向于选择“间隔最大”的边界。

二维图中是一条线；本 notebook 后面的设备故障数据有多个特征，所以 SVM 找到的是一个高维空间中的超平面。

## 4. 读取工业设备数据

In [ ]:
csv_path = Path.cwd() / "predictive_maintenance.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day31-45/predictive_maintenance.csv")

df = pd.read_csv(csv_path)
df.head()

## 5. 定义特征和目标变量

In [ ]:
target = "Machine failure"
failure_mode_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]

numeric_features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]
categorical_features = ["Type"]

X = df[numeric_features + categorical_features]
y = df[target]

## 6. 检查目标变量分布

这个数据集明显类别不平衡：故障样本远少于正常样本。因此不能只看准确率，还要重点关注故障类别的 Recall、Precision 和 F1。

In [ ]:
print("数据形状:", df.shape)
print("缺失值数量:", int(df.isna().sum().sum()))
print("\n目标变量分布:")
print(y.value_counts().rename(index={0: "正常", 1: "故障"}))
print(f"\n故障比例: {y.mean():.2%}")

In [ ]:
df[numeric_features].describe().T

## 7. 拆分训练集和测试集

使用 `stratify=y` 保持训练集和测试集中的故障比例接近原数据。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("训练集:", X_train.shape)
print("测试集:", X_test.shape)
print("训练集故障比例:", f"{y_train.mean():.2%}")
print("测试集故障比例:", f"{y_test.mean():.2%}")

## 8. 建立预处理流程

SVM 对特征尺度非常敏感，因此数值特征必须标准化。

`Type` 是类别特征，使用 One-Hot 编码转换成 0/1 数值列。

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

## 9. 训练线性 SVM

先使用：

- `kernel="linear"`：寻找线性超平面；
- `C=1.0`：常见的默认惩罚强度；
- `class_weight="balanced"`：根据类别频率自动调整误分类代价，缓解类别不平衡。

In [ ]:
svm = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            SVC(
                kernel="linear",
                C=1.0,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

svm.fit(X_train, y_train)

## 10. 测试集评估

In [ ]:
y_pred = svm.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["正常", "故障"], digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["正常", "故障"],
    cmap="Blues",
    values_format="d",
)
plt.title("Linear SVM 混淆矩阵")
plt.show()

## 11. 比较是否使用类别权重

由于故障样本很少，普通 SVM 可能更倾向于把样本预测为“正常”，因为这样准确率更高。

`class_weight="balanced"` 会提高误分类少数类的代价，让模型更重视故障样本。

In [ ]:
rows = []

for class_weight in [None, "balanced"]:
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                SVC(
                    kernel="linear",
                    C=1.0,
                    class_weight=class_weight,
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rows.append(
        {
            "class_weight": str(class_weight),
            "accuracy": model.score(X_test, y_test),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

class_weight_results = pd.DataFrame(rows)
class_weight_results

## 12. 调整参数 C

下面继续固定 `class_weight="balanced"`，比较不同 `C` 值。

重点关注故障类别的 F1，以及工业场景中很重要的 Recall。

In [ ]:
c_values = [0.01, 0.1, 1.0, 10.0, 100.0]
rows = []

for c in c_values:
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            (
                "model",
                SVC(
                    kernel="linear",
                    C=c,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rows.append(
        {
            "C": c,
            "accuracy": model.score(X_test, y_test),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

c_results = pd.DataFrame(rows)
c_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(c_results["C"], c_results["precision"], marker="o", label="Precision")
plt.plot(c_results["C"], c_results["recall"], marker="o", label="Recall")
plt.plot(c_results["C"], c_results["f1"], marker="o", label="F1")
plt.xscale("log")
plt.xlabel("C (log scale)")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("不同 C 值下的测试集表现")
plt.show()

In [ ]:
best_row = c_results.sort_values("f1", ascending=False).iloc[0]
best_c = float(best_row["C"])

best_svm = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            SVC(
                kernel="linear",
                C=best_c,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)
best_svm.fit(X_train, y_train)

print(f"按 F1 选择的 C: {best_c}")
print(f"对应 F1: {best_row['f1']:.3f}")

## 13. 移动分界线：调整 SVM 决策阈值

SVM 默认使用阈值 `0`：

- `decision_function > 0`：预测为故障；
- `decision_function < 0`：预测为正常。

调整阈值相当于移动分界线：

- 降低阈值：更多样本被判断为故障，Recall 通常会上升，Precision 可能下降；
- 提高阈值：更少样本被判断为故障，Precision 可能上升，Recall 可能下降。

In [ ]:
decision_scores = best_svm.decision_function(X_test)

rows = []
for threshold in [-0.5, 0.0, 0.5]:
    pred = (decision_scores >= threshold).astype(int)
    rows.append(
        {
            "threshold": threshold,
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

threshold_results = pd.DataFrame(rows)
threshold_results

## 14. 预测新样本

`decision_function` 的绝对值表示样本离 SVM 分界面的距离，符号表示位于分界面的哪一侧。

In [ ]:
new_sample = pd.DataFrame(
    [
        {
            "Type": "M",
            "Air temperature": 299.0,
            "Process temperature": 311.0,
            "Rotational speed": 1650,
            "Torque": 55.0,
            "Tool wear": 180,
        }
    ]
)

predicted_class = best_svm.predict(new_sample)[0]
decision_score = best_svm.decision_function(new_sample)[0]

print("预测结果:", "故障" if predicted_class == 1 else "正常")
print(f"decision_function: {decision_score:.3f}")
new_sample

## 15. 总结

本 notebook 演示了 SVM 的基本使用方式：

- SVM 寻找的是间隔尽可能大的分类边界；
- 离边界最近的样本是支持向量；
- 二维中边界是一条线，高维中是一个超平面；
- `C` 控制“间隔大小”和“训练误分类代价”之间的平衡；
- SVM 对特征尺度敏感，数值特征必须标准化；
- 类别特征需要先进行 One-Hot 编码；
- 类别不平衡时，`class_weight="balanced"` 和决策阈值调整非常常用；
- 对设备故障预测任务，应重点关注故障类别的 Recall、Precision 和 F1。

## 可以进一步尝试

1. 使用 `GridSearchCV` 系统搜索 `C`；
2. 尝试 `kernel="rbf"`，处理非线性边界；
3. 比较 SVM、逻辑回归、KNN 和随机森林；
4. 尝试少数类过采样方法，如 SMOTE；
5. 按业务成本选择阈值，在漏报和误报之间做权衡。